# Experimental Results Analysis: Parameter Study

## Overview
Analysis of experimental results with:
- **Features**: T_1, T_2, Q_1, Q_2 (from config profiles)
- **Targets**: overall_accuracy, and per-class accuracies (entailment, non-entailment, not_mentioned, uncertain)

## Fractional Factorial Design Considerations
With 4 factors, this problem is well-suited for fractional factorial design:
- Full factorial (2 levels): 2^4 = 16 runs
- Half-fraction: 2^(4-1) = 8 runs
- Quarter-fraction: 2^(4-2) = 4 runs

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import cross_val_score

# Design of Experiments (DOE)
try:
    import pyDOE2 as pyDOE
    PYDOE_AVAILABLE = True
except ImportError:
    PYDOE_AVAILABLE = False
    print("pyDOE2 not available. Install with: pip install pyDOE2")

try:
    import statsmodels.api as sm
    from statsmodels.formula.api import ols
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("statsmodels not available. Install with: pip install statsmodels")

plt.style.use('seaborn-v0_8-whitegrid')
print("Imports successful!")

## 1. Load Configuration and Results Data

Adjust the paths below to match your data structure.

In [ ]:
# ============================================================
# CONFIGURATION - MODIFY THESE PATHS FOR YOUR DATA
# ============================================================

# Path to directory containing config files (config1.json, config2.json, etc.)
CONFIG_DIR = Path("./profiles")  # or "./configs" or wherever your configs are

# Path to results file (CSV or JSON)
RESULTS_PATH = Path("./results.csv")  # or "./results.json"

# Feature column names (parameters from configs)
FEATURE_COLS = ['T_1', 'T_2', 'Q_1', 'Q_2']

# Target column names (accuracies)
TARGET_COLS = [
    'overall_accuracy',
    'accuracy_entailment',
    'accuracy_non_entailment', 
    'accuracy_not_mentioned',
    'accuracy_uncertain'
]

In [ ]:
def load_configs_from_profiles(config_dir: Path) -> pd.DataFrame:
    """
    Load parameters from config files (config1.json, config2.json, etc.)
    
    Expected structure:
    {
        "profile": {
            "T_1": value,
            "T_2": value,
            "Q_1": value,
            "Q_2": value
        }
    }
    """
    configs = []
    config_files = sorted(config_dir.glob("config*.json"))
    
    for config_file in config_files:
        with open(config_file, 'r') as f:
            data = json.load(f)
        
        # Extract profile section (adjust key names as needed)
        profile = data.get('profile', data)  # fallback to root if no 'profile' key
        
        config_entry = {
            'config_id': config_file.stem,
            'T_1': profile.get('T_1'),
            'T_2': profile.get('T_2'),
            'Q_1': profile.get('Q_1'),
            'Q_2': profile.get('Q_2'),
        }
        configs.append(config_entry)
    
    return pd.DataFrame(configs)


def load_results(results_path: Path) -> pd.DataFrame:
    """
    Load results from CSV or JSON file.
    """
    if results_path.suffix == '.csv':
        return pd.read_csv(results_path)
    elif results_path.suffix == '.json':
        with open(results_path, 'r') as f:
            data = json.load(f)
        return pd.DataFrame(data)
    else:
        raise ValueError(f"Unsupported file format: {results_path.suffix}")

In [ ]:
# ============================================================
# OPTION A: Load from separate config files and results file
# ============================================================

# Uncomment and use if you have separate files:
# df_configs = load_configs_from_profiles(CONFIG_DIR)
# df_results = load_results(RESULTS_PATH)
# df = df_configs.merge(df_results, on='config_id')

# ============================================================
# OPTION B: Load from single combined CSV/JSON
# ============================================================

# df = pd.read_csv("your_combined_data.csv")

# ============================================================
# OPTION C: Use demo data to test the notebook
# ============================================================

# Generate synthetic demo data (2^4 factorial design)
np.random.seed(42)

# Create 2-level factorial design
levels = [-1, 1]  # coded levels
from itertools import product
factorial_design = list(product(levels, repeat=4))

df = pd.DataFrame(factorial_design, columns=FEATURE_COLS)

# Add config IDs
df['config_id'] = [f'config{i+1}' for i in range(len(df))]

# Generate synthetic targets with known effects
# True model: accuracy depends on T_1, Q_1, and T_1*Q_2 interaction
df['overall_accuracy'] = (
    0.75 + 
    0.05 * df['T_1'] + 
    0.03 * df['Q_1'] + 
    0.02 * df['T_1'] * df['Q_2'] +
    np.random.normal(0, 0.01, len(df))
)

df['accuracy_entailment'] = df['overall_accuracy'] + np.random.normal(0.02, 0.02, len(df))
df['accuracy_non_entailment'] = df['overall_accuracy'] + np.random.normal(-0.01, 0.02, len(df))
df['accuracy_not_mentioned'] = df['overall_accuracy'] + np.random.normal(-0.03, 0.03, len(df))
df['accuracy_uncertain'] = df['overall_accuracy'] + np.random.normal(-0.05, 0.04, len(df))

# Clip to valid range
for col in TARGET_COLS:
    df[col] = df[col].clip(0, 1)

print(f"Loaded {len(df)} experimental runs")
print(f"Features: {FEATURE_COLS}")
print(f"Targets: {TARGET_COLS}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

print("\n--- Feature Statistics ---")
display(df[FEATURE_COLS].describe())

print("\n--- Target Statistics ---")
display(df[TARGET_COLS].describe())

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature-Target correlations
corr_ft = df[FEATURE_COLS + TARGET_COLS].corr()
sns.heatmap(
    corr_ft.loc[FEATURE_COLS, TARGET_COLS], 
    annot=True, 
    cmap='RdBu_r', 
    center=0,
    ax=axes[0],
    vmin=-1, vmax=1
)
axes[0].set_title('Feature-Target Correlations')

# Target-Target correlations
corr_tt = df[TARGET_COLS].corr()
sns.heatmap(
    corr_tt, 
    annot=True, 
    cmap='RdBu_r', 
    center=0,
    ax=axes[1],
    vmin=-1, vmax=1
)
axes[1].set_title('Target-Target Correlations')

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of accuracy metrics
fig, axes = plt.subplots(1, len(TARGET_COLS), figsize=(4*len(TARGET_COLS), 4))

for i, target in enumerate(TARGET_COLS):
    axes[i].hist(df[target], bins=15, edgecolor='black', alpha=0.7)
    axes[i].axvline(df[target].mean(), color='red', linestyle='--', label=f'Mean: {df[target].mean():.3f}')
    axes[i].set_xlabel(target)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Distribution of Accuracy Metrics', y=1.02)
plt.tight_layout()
plt.show()

## 3. Fractional Factorial Design Analysis

### Checking if Your Design Fits a Fractional Factorial Structure

In [ ]:
def analyze_design_structure(df, feature_cols):
    """
    Analyze if the experimental design fits a factorial or fractional factorial structure.
    """
    print("=" * 60)
    print("DESIGN STRUCTURE ANALYSIS")
    print("=" * 60)
    
    n_runs = len(df)
    n_factors = len(feature_cols)
    
    print(f"\nNumber of experimental runs: {n_runs}")
    print(f"Number of factors: {n_factors}")
    
    # Check levels per factor
    print("\n--- Factor Levels ---")
    levels_per_factor = {}
    for col in feature_cols:
        unique_vals = sorted(df[col].unique())
        levels_per_factor[col] = len(unique_vals)
        print(f"{col}: {len(unique_vals)} levels -> {unique_vals}")
    
    # Check if it's a 2-level design
    all_two_level = all(n == 2 for n in levels_per_factor.values())
    
    if all_two_level:
        full_factorial_runs = 2 ** n_factors
        print(f"\n--- 2-Level Design Analysis ---")
        print(f"Full factorial would require: 2^{n_factors} = {full_factorial_runs} runs")
        print(f"Your design has: {n_runs} runs")
        
        if n_runs == full_factorial_runs:
            print("✓ This is a FULL FACTORIAL design (2^k)")
            design_type = 'full_factorial'
            resolution = 'Full'
        elif n_runs == full_factorial_runs // 2:
            print(f"✓ This is a HALF-FRACTION design (2^({n_factors}-1))")
            design_type = 'fractional'
            resolution = 'IV or V'
        elif n_runs == full_factorial_runs // 4:
            print(f"✓ This is a QUARTER-FRACTION design (2^({n_factors}-2))")
            design_type = 'fractional'
            resolution = 'III or IV'
        else:
            print(f"⚠ Non-standard fraction: {n_runs}/{full_factorial_runs}")
            design_type = 'other'
            resolution = 'Unknown'
            
        return {
            'type': design_type,
            'resolution': resolution,
            'n_runs': n_runs,
            'n_factors': n_factors,
            'levels': levels_per_factor,
            'is_two_level': True
        }
    else:
        print("\n⚠ Mixed-level design detected (not pure 2-level)")
        print("Consider: Response Surface Methodology (RSM) or general factorial analysis")
        return {
            'type': 'mixed_level',
            'n_runs': n_runs,
            'n_factors': n_factors,
            'levels': levels_per_factor,
            'is_two_level': False
        }

design_info = analyze_design_structure(df, FEATURE_COLS)

In [ ]:
# Check orthogonality of design
def check_orthogonality(df, feature_cols):
    """
    Check if the design matrix is orthogonal (important for factorial designs).
    In an orthogonal design, correlations between factors should be ~0.
    """
    print("\n=" * 60)
    print("ORTHOGONALITY CHECK")
    print("=" * 60)
    
    X = df[feature_cols].values
    
    # Standardize if needed
    X_centered = X - X.mean(axis=0)
    
    # Compute correlation matrix
    corr_matrix = np.corrcoef(X_centered.T)
    
    # Check off-diagonal elements
    mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
    off_diag = corr_matrix[mask]
    max_corr = np.abs(off_diag).max()
    
    print(f"\nMaximum off-diagonal correlation: {max_corr:.4f}")
    
    if max_corr < 0.01:
        print("✓ Design is orthogonal (correlations ≈ 0)")
    elif max_corr < 0.1:
        print("⚠ Design is nearly orthogonal")
    else:
        print("✗ Design is NOT orthogonal - factors are correlated")
        print("  This may indicate confounding or an irregular design")
    
    # Display correlation matrix
    print("\nFactor Correlation Matrix:")
    corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)
    display(corr_df.round(4))
    
    return max_corr < 0.1

is_orthogonal = check_orthogonality(df, FEATURE_COLS)

## 4. Main Effects Analysis

In [ ]:
def compute_main_effects(df, feature_cols, target_col):
    """
    Compute main effects for each factor.
    Main effect = (mean at high level) - (mean at low level)
    """
    effects = {}
    
    for factor in feature_cols:
        levels = sorted(df[factor].unique())
        if len(levels) == 2:
            low, high = levels
            mean_low = df[df[factor] == low][target_col].mean()
            mean_high = df[df[factor] == high][target_col].mean()
            effects[factor] = mean_high - mean_low
        else:
            # For multi-level factors, use regression coefficient
            effects[factor] = np.nan
    
    return effects

# Compute main effects for all targets
print("=" * 60)
print("MAIN EFFECTS ANALYSIS")
print("=" * 60)

effects_df = pd.DataFrame()

for target in TARGET_COLS:
    effects = compute_main_effects(df, FEATURE_COLS, target)
    effects_df[target] = pd.Series(effects)

print("\nMain Effects (High - Low):")
display(effects_df.round(4))

In [ ]:
# Visualize main effects
fig, axes = plt.subplots(1, len(TARGET_COLS), figsize=(4*len(TARGET_COLS), 5))

for i, target in enumerate(TARGET_COLS):
    effects = effects_df[target]
    colors = ['green' if e > 0 else 'red' for e in effects]
    
    bars = axes[i].barh(effects.index, effects.values, color=colors, alpha=0.7)
    axes[i].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    axes[i].set_xlabel('Effect Size')
    axes[i].set_title(f'Main Effects: {target}')
    
    # Add value labels
    for bar, val in zip(bars, effects.values):
        axes[i].text(val, bar.get_y() + bar.get_height()/2, 
                    f' {val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Interaction Effects Analysis

In [ ]:
def compute_two_way_interactions(df, feature_cols, target_col):
    """
    Compute two-way interaction effects for 2-level factors.
    """
    from itertools import combinations
    
    interactions = {}
    
    for f1, f2 in combinations(feature_cols, 2):
        levels_f1 = sorted(df[f1].unique())
        levels_f2 = sorted(df[f2].unique())
        
        if len(levels_f1) == 2 and len(levels_f2) == 2:
            # Create interaction column
            interaction_col = df[f1] * df[f2]
            
            low = interaction_col.min()
            high = interaction_col.max()
            
            mean_low = df[interaction_col == low][target_col].mean()
            mean_high = df[interaction_col == high][target_col].mean()
            
            interactions[f"{f1}:{f2}"] = mean_high - mean_low
    
    return interactions

# Compute interactions for overall accuracy
print("=" * 60)
print("TWO-WAY INTERACTION EFFECTS")
print("=" * 60)

interactions_df = pd.DataFrame()

for target in TARGET_COLS:
    interactions = compute_two_way_interactions(df, FEATURE_COLS, target)
    interactions_df[target] = pd.Series(interactions)

print("\nTwo-Way Interaction Effects:")
display(interactions_df.round(4))

In [ ]:
# Interaction plots for overall accuracy
from itertools import combinations

factor_pairs = list(combinations(FEATURE_COLS, 2))
n_pairs = len(factor_pairs)
n_cols = 3
n_rows = (n_pairs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

target = 'overall_accuracy'

for i, (f1, f2) in enumerate(factor_pairs):
    ax = axes[i]
    
    # Group by both factors
    grouped = df.groupby([f1, f2])[target].mean().unstack()
    
    for col in grouped.columns:
        ax.plot(grouped.index, grouped[col], marker='o', label=f'{f2}={col}')
    
    ax.set_xlabel(f1)
    ax.set_ylabel(target)
    ax.set_title(f'{f1} × {f2} Interaction')
    ax.legend(title=f2)

# Hide empty subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Interaction Plots for {target}', y=1.02)
plt.tight_layout()
plt.show()

## 6. Regression Analysis with Interactions

In [ ]:
def fit_factorial_model(df, feature_cols, target_col, include_interactions=True):
    """
    Fit a linear regression model with main effects and optional interactions.
    """
    X = df[feature_cols].values
    y = df[target_col].values
    
    # Create polynomial features (main effects + interactions)
    if include_interactions:
        poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
        X_poly = poly.fit_transform(X)
        feature_names = poly.get_feature_names_out(feature_cols)
    else:
        X_poly = X
        feature_names = feature_cols
    
    # Fit model
    model = LinearRegression()
    model.fit(X_poly, y)
    
    # Compute R-squared
    r2 = model.score(X_poly, y)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_poly, y, cv=min(5, len(df)), scoring='r2')
    
    # Coefficients
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    
    return {
        'model': model,
        'r2': r2,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'coefficients': coef_df,
        'intercept': model.intercept_
    }

# Fit models for all targets
print("=" * 60)
print("REGRESSION ANALYSIS WITH INTERACTIONS")
print("=" * 60)

for target in TARGET_COLS:
    print(f"\n--- {target} ---")
    
    result = fit_factorial_model(df, FEATURE_COLS, target, include_interactions=True)
    
    print(f"R² = {result['r2']:.4f}")
    print(f"CV R² = {result['cv_r2_mean']:.4f} ± {result['cv_r2_std']:.4f}")
    print(f"Intercept = {result['intercept']:.4f}")
    print("\nCoefficients (sorted by magnitude):")
    display(result['coefficients'].round(4))

In [ ]:
# ANOVA-style analysis using statsmodels (if available)
if STATSMODELS_AVAILABLE:
    print("=" * 60)
    print("ANOVA-STYLE ANALYSIS")
    print("=" * 60)
    
    target = 'overall_accuracy'
    
    # Create formula with all main effects and 2-way interactions
    main_effects = ' + '.join(FEATURE_COLS)
    interactions = ' + '.join([f"{f1}:{f2}" for f1, f2 in combinations(FEATURE_COLS, 2)])
    formula = f"{target} ~ {main_effects} + {interactions}"
    
    print(f"\nFormula: {formula}")
    
    # Fit OLS model
    model = ols(formula, data=df).fit()
    
    print("\n--- Model Summary ---")
    print(model.summary())
    
    print("\n--- ANOVA Table ---")
    anova_table = sm.stats.anova_lm(model, typ=2)
    display(anova_table.round(4))
else:
    print("Install statsmodels for ANOVA analysis: pip install statsmodels")

## 7. Per-Class Accuracy Analysis

In [ ]:
# Compare per-class accuracies across experimental conditions
class_targets = [t for t in TARGET_COLS if t != 'overall_accuracy']

print("=" * 60)
print("PER-CLASS ACCURACY COMPARISON")
print("=" * 60)

# Melt data for visualization
df_melted = df.melt(
    id_vars=['config_id'] + FEATURE_COLS,
    value_vars=class_targets,
    var_name='class',
    value_name='accuracy'
)
df_melted['class'] = df_melted['class'].str.replace('accuracy_', '')

# Summary by class
class_summary = df_melted.groupby('class')['accuracy'].agg(['mean', 'std', 'min', 'max'])
print("\nAccuracy by Class:")
display(class_summary.round(4))

In [ ]:
# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=df_melted, x='class', y='accuracy', ax=ax, palette='Set2')
sns.stripplot(data=df_melted, x='class', y='accuracy', ax=ax, color='black', alpha=0.5, size=4)

ax.set_xlabel('Class')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Distribution by Class')

plt.tight_layout()
plt.show()

In [ ]:
# Effect of each factor on per-class accuracies
fig, axes = plt.subplots(len(FEATURE_COLS), 1, figsize=(12, 4*len(FEATURE_COLS)))

for i, factor in enumerate(FEATURE_COLS):
    ax = axes[i]
    
    # Group by factor level and class
    grouped = df_melted.groupby([factor, 'class'])['accuracy'].mean().unstack()
    
    x = np.arange(len(grouped.columns))
    width = 0.35
    
    levels = sorted(df[factor].unique())
    for j, level in enumerate(levels):
        offset = (j - 0.5) * width
        ax.bar(x + offset, grouped.loc[level], width, label=f'{factor}={level}')
    
    ax.set_xticks(x)
    ax.set_xticklabels(grouped.columns, rotation=45, ha='right')
    ax.set_ylabel('Mean Accuracy')
    ax.set_title(f'Effect of {factor} on Per-Class Accuracy')
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Optimal Configuration Finding

In [ ]:
# Find best configurations for each target
print("=" * 60)
print("OPTIMAL CONFIGURATIONS")
print("=" * 60)

for target in TARGET_COLS:
    best_idx = df[target].idxmax()
    best_row = df.loc[best_idx]
    
    print(f"\n--- Best for {target} ---")
    print(f"Config: {best_row['config_id']}")
    print(f"Value: {best_row[target]:.4f}")
    print(f"Parameters: {dict(best_row[FEATURE_COLS])}")

In [ ]:
# Pareto front analysis (trade-off between different objectives)
def find_pareto_front(df, objectives, minimize=False):
    """
    Find Pareto-optimal configurations.
    """
    is_pareto = np.ones(len(df), dtype=bool)
    values = df[objectives].values
    
    if minimize:
        values = -values
    
    for i in range(len(df)):
        for j in range(len(df)):
            if i != j:
                # Check if j dominates i
                if np.all(values[j] >= values[i]) and np.any(values[j] > values[i]):
                    is_pareto[i] = False
                    break
    
    return is_pareto

# Find Pareto front for overall vs. uncertain accuracy
pareto_objectives = ['overall_accuracy', 'accuracy_uncertain']
df['is_pareto'] = find_pareto_front(df, pareto_objectives)

print("\n--- Pareto-Optimal Configurations ---")
pareto_configs = df[df['is_pareto']][['config_id'] + FEATURE_COLS + pareto_objectives]
display(pareto_configs.round(4))

# Visualize Pareto front
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    df[~df['is_pareto']][pareto_objectives[0]], 
    df[~df['is_pareto']][pareto_objectives[1]],
    alpha=0.5, label='Dominated', s=100
)
ax.scatter(
    df[df['is_pareto']][pareto_objectives[0]], 
    df[df['is_pareto']][pareto_objectives[1]],
    color='red', marker='*', s=200, label='Pareto Front'
)

# Add labels
for _, row in df[df['is_pareto']].iterrows():
    ax.annotate(row['config_id'], (row[pareto_objectives[0]], row[pareto_objectives[1]]),
               textcoords="offset points", xytext=(5,5), fontsize=8)

ax.set_xlabel(pareto_objectives[0])
ax.set_ylabel(pareto_objectives[1])
ax.set_title('Pareto Front: Overall vs. Uncertain Accuracy')
ax.legend()

plt.tight_layout()
plt.show()

## 9. Summary and Recommendations

In [ ]:
print("="*60)
print("ANALYSIS SUMMARY")
print("="*60)

print(f"""
EXPERIMENTAL DESIGN:
- Number of runs: {len(df)}
- Number of factors: {len(FEATURE_COLS)}
- Design type: {design_info['type']}
- Orthogonal: {'Yes' if is_orthogonal else 'No'}

KEY FINDINGS:
""")

# Identify most important factors
main_effects_overall = effects_df['overall_accuracy']
most_important = main_effects_overall.abs().sort_values(ascending=False)

print("Most influential factors (by main effect magnitude):")
for factor, effect in most_important.items():
    direction = "↑" if main_effects_overall[factor] > 0 else "↓"
    print(f"  {factor}: {direction} {abs(effect):.4f}")

# Identify significant interactions
interactions_overall = interactions_df['overall_accuracy']
significant_interactions = interactions_overall[interactions_overall.abs() > 0.01]

if len(significant_interactions) > 0:
    print("\nNotable interactions:")
    for interaction, effect in significant_interactions.abs().sort_values(ascending=False).items():
        print(f"  {interaction}: {interactions_overall[interaction]:.4f}")

print(f"""
RECOMMENDATIONS:
1. Focus on tuning: {most_important.index[0]} (largest main effect)
2. Consider interaction: {interactions_overall.abs().idxmax()} (largest interaction)
3. Best overall config: {df.loc[df['overall_accuracy'].idxmax(), 'config_id']}
""")

In [ ]:
# Export results
print("Saving analysis results...")

# Save main effects
effects_df.to_csv('main_effects.csv')

# Save interactions
interactions_df.to_csv('interaction_effects.csv')

# Save full data with analysis
df.to_csv('analyzed_results.csv', index=False)

print("✓ Saved: main_effects.csv")
print("✓ Saved: interaction_effects.csv")
print("✓ Saved: analyzed_results.csv")